In [ ]:
import glob
import rasterio
from rasterio.merge import merge
import geopandas as gpd
from rasterio.mask import mask

In [ ]:
#Gather all individual country binary rasters
file_list = glob.glob(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\INDONESIA\By Island\*.tif")
src_files_to_mosaic = [rasterio.open(fp) for fp in file_list]
# 2. Merge rasters spatially into a single regional raster
# Using method='max' ensures that if country borders overlap, presence (1) overrides absence (0)
mosaic, out_transform = merge(
    src_files_to_mosaic, 
    method='max', 
    nodata=0
)
# 3. Update metadata to reflect the expanded regional extent and grid resolution
out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "nodata": 0,
    "compress": "lzw"
})

In [ ]:
with rasterio.open(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\INDONESIA\By Island\Mosaic_Rubber_Indonesia_2024.tif", "w", **out_meta) as dest:
    dest.write(mosaic)
#Close open file connections
for src in src_files_to_mosaic:
    src.close()

In [ ]:
# 1) Read AOI shapefile
aoi = gpd.read_file(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\INDONESIA\INDO_AOI.shp")

# 2) Make sure CRS matches the raster
# Use the mosaic file's CRS or the raster you created
with rasterio.open(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\INDONESIA\Final Mosaic\Mosaic_Rubber_Indonesia_2024.tif") as src:
    raster_crs = src.crs
    print("Raster CRS:", raster_crs)

aoi = aoi.to_crs(raster_crs)

# 3) Clip the raster to AOI geometry
with rasterio.open(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\INDONESIA\Final Mosaic\Mosaic_Rubber_Indonesia_2024.tif") as src:
    shapes = [geom for geom in aoi.geometry]
    clipped, out_transform = mask(
        src,
        shapes=shapes,
        crop=True,
        nodata=0,
        all_touched=False
    )

    out_meta = src.meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": clipped.shape[1],
        "width": clipped.shape[2],
        "transform": out_transform,
        "nodata": 0,
        "compress": "lzw"
    })

# 4) Save clipped output
with rasterio.open(
    r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\INDONESIA\Final Mosaic\Mosaic_Rubber_Indonesia_2024_clipped.tif",
    "w",
    **out_meta
) as dest:
    dest.write(clipped)